In [2]:
import pandas as pd
import numpy as np
import glob
import os
# save_path = "logs/New_folder/logs/*"
# dfs = []
# for file in glob.glob(save_path):
#     df = pd.read_csv(file)
#     # df["model_name"] = df["model_name"] + "_" + os.path.basename(file)
#     dfs.append(df)
# merged_df = pd.concat(dfs, ignore_index=True)
# # merged_df.to_csv("logs/New_folder/merged_data.csv", index=False)
# pd.set_option("display.precision", 3)
# merged_df.head()
# merged_df["model_name"].unique()

In [31]:
df = pd.read_csv("logs/New_folder/logs/eval_log.csv")
df.head()

,episode,t,hed_action,hed_share,stock_price,stock_position,stock_pnl,liab_port_gamma,liab_port_vega,liab_port_pnl,...,gamma_after_hedge,vega_before_hedge,vega_after_hedge,step_pnl,state_price,state_gamma,state_vega,state_hed_gamma,state_hed_vega,model_name
0,0,0,0.967901,1.366995,10.000000,-18.253853,2.490793,-27.247644,-1.941457,7.734381,...,25.498387,-1.941457,-0.062319,-1.323861,9.863547,-1.292742,-2.000396,39.319261,1.355909,ckpt_11600.pth
1,0,1,0.973336,1.435979,9.863547,-36.466896,6.264425,-55.833177,-3.818508,13.087833,...,55.168888,-2.000396,-0.053339,-1.417767,9.691763,53.763471,-0.128796,38.430841,1.332147,ckpt_11600.pth
2,0,2,0.924242,-0.016625,9.691763,-26.269669,-8.367946,-54.842295,-3.706997,-30.371029,...,53.124568,-0.128796,-0.150942,1.723513,10.010303,54.448144,-0.142389,37.518486,1.375963,ckpt_11600.pth
3,0,3,0.913646,-0.030773,10.010303,-41.803388,-15.338183,-53.785339,-3.780360,-42.362051,...,53.293599,-0.142389,-0.184731,2.556392,10.377216,70.774232,1.632928,36.376048,1.426417,ckpt_11600.pth
4,0,4,0.921891,-0.151970,10.377216,-105.578104,-18.101999,-23.750308,-1.635776,-11.693622,...,65.246164,1.632928,1.416156,-1.464448,10.548672,59.611134,1.220042,36.819185,1.450097,ckpt_11600.pth


In [32]:
# df = pd.read_csv("logs/New_folder/logs/eval_log.csv")
episode_pnl = (
    df.groupby(["model_name", "episode"])["step_pnl"].sum()
      .rename("total_pnl")
)
episode_hed_cost = (
    df.groupby(["model_name", "episode"])["hed_cost"].sum()
      .rename("total_hed_cost")
)

In [33]:
sign_gamma = np.sign(df["gamma_before_hedge"])
sign_vega = np.sign(df["vega_before_hedge"])
df["gamma_hedged_pct"] = 1 - (sign_gamma* df.gamma_after_hedge) / df.gamma_before_hedge

df["vega_hedged_pct"] = 1 - (sign_vega*df.vega_after_hedge) / df.vega_before_hedge

def hedging_ratio_episode(df_ep):
    num = (np.sign(df_ep.gamma_before_hedge) * df_ep.gamma_after_hedge).sum()
    den = (np.sign(df_ep.gamma_before_hedge) * df_ep.gamma_before_hedge).sum()
    if np.isclose(den, 0):
        return np.nan
    return 1 - num / den
def vega_hedging_ratio_episode(df_ep):
    num = (np.sign(df_ep.vega_before_hedge) * df_ep.vega_after_hedge).sum()
    den = (np.sign(df_ep.vega_before_hedge) * df_ep.vega_before_hedge).sum()
    if np.isclose(den, 0):
        return np.nan
    return 1 - num / den
gamma_hr = df.groupby(["model_name", "episode"]).apply(hedging_ratio_episode).rename("gamma_hedge_ratio")
vega_hr = df.groupby(["model_name", "episode"]).apply(vega_hedging_ratio_episode).rename("vega_hedge_ratio")
result = pd.concat([gamma_hr, vega_hr, episode_pnl, episode_hed_cost], axis=1).reset_index()  #result is the P&L per episode per model and concat is done by index, all have multiindex
model_summary = (
    result.groupby("model_name",as_index=False)
          .agg({
               "gamma_hedge_ratio": "mean",
               "vega_hedge_ratio": "mean",
               "total_pnl": ["mean", "std"],
               "total_hed_cost": "mean"
          })
)

C:\Users\sunbo\AppData\Local\Temp\ipykernel_27388\3571176183.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  gamma_hr = df.groupby(["model_name", "episode"]).apply(hedging_ratio_episode).rename("gamma_hedge_ratio")
C:\Users\sunbo\AppData\Local\Temp\ipykernel_27388\3571176183.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  vega_hr = df.groupby(["model_name", "episode"]).apply(vega_hedging_ratio_epi

In [105]:
# path = "logs/New_folder/merged_data.csv"
# df = pd.read_csv(path)
result.to_csv("logs/New_folder/result_episodes.csv",index=False)
result.head()

,model_name,episode,gamma_hedge_ratio,vega_hedge_ratio,total_pnl,total_hed_cost
0,11600.pth_weightedmean,0,0.092,0.033,2.496,-3.621
1,11600.pth_weightedmean,1,0.160,0.051,12.752,-6.038
2,11600.pth_weightedmean,2,0.098,0.019,-21.350,-9.524
3,11600.pth_weightedmean,3,0.205,0.100,3.742,-4.791
4,11600.pth_weightedmean,4,0.283,0.016,-13.681,-8.294


In [34]:
episode_pnl = (
    df.groupby(["model_name", "episode"])["step_pnl"]
      .sum()
      .reset_index()
      .rename(columns={"step_pnl": "episode_pnl"})
)

def compute_var(x, alpha=0.05):
    return np.quantile(x, alpha)

def compute_cvar(x, alpha=0.05):
    var = compute_var(x, alpha)
    return x[x <= var].mean()

model_results = episode_pnl.groupby("model_name")["episode_pnl"].agg(
    VaR_5=lambda x: compute_var(x, 0.05),
    CVaR_5=lambda x: compute_cvar(x, 0.05),
).reset_index()

print(model_results.sort_values("VaR_5"))

       model_name      VaR_5     CVaR_5
1  ckpt_15400.pth -51.742125 -67.476635
4  ckpt_35000.pth -50.650660 -68.867120
3  ckpt_27800.pth -45.710847 -62.447906
0  ckpt_11600.pth -44.107757 -61.418976
2  ckpt_18400.pth -37.891301 -51.615538


In [35]:
model_summary.columns = [
    f"{a}_{b}" if b != "" else a
    for a, b in model_summary.columns
]
model_summary = model_summary.rename(columns={
    'gamma_hedge_ratio_mean': 'gamma_hr',
    'vega_hedge_ratio_mean': 'vega_hr',
    'total_pnl_mean': 'pnl_mean',
    'total_pnl_std': 'pnl_std',
    'total_hed_cost_mean': 'hed_cost',
})
model_summary["meanstd"] = model_summary["pnl_mean"] - 1.645 * model_summary["pnl_std"]
model_summary

,model_name,gamma_hr,vega_hr,pnl_mean,pnl_std,hed_cost,meanstd
0,ckpt_11600.pth,0.068322,0.181171,-11.504211,21.686119,-11.153907,-47.177877
1,ckpt_15400.pth,0.159692,0.208512,-16.668162,22.248895,-16.363341,-53.267594
2,ckpt_18400.pth,0.232786,0.097011,-6.319413,19.155265,-6.307952,-37.829824
3,ckpt_27800.pth,0.165469,0.191149,-13.884539,20.202299,-13.526777,-47.117320
4,ckpt_35000.pth,0.055482,0.213707,-15.203729,22.927166,-14.915264,-52.918917


In [37]:
merged_df = pd.merge(model_summary, model_results, on="model_name", how="left") 
merged_df["model_name"] = merged_df["model_name"].apply(lambda x: "_".join(x.split("_")[1:3]).replace(".csv",""))
merged_df = merged_df.sort_values("VaR_5",ascending=False).reset_index(drop = True)
index_map = {
    "11600.pth": "Weighted Mean",
    "15400.pth": "CVaR",
    "18400.pth": "Mean-CVaR",
    "35000.pth": "Mean-Std",
    "27800.pth": "VaR",

}
merged_df.set_index("model_name",inplace=True)
merged_df = merged_df.rename(index=index_map)
merged_df

,gamma_hr,vega_hr,pnl_mean,pnl_std,hed_cost,meanstd,VaR_5,CVaR_5
model_name,,,,,,,,
Mean-CVaR,0.232786,0.097011,-6.319413,19.155265,-6.307952,-37.829824,-37.891301,-51.615538
Weighted Mean,0.068322,0.181171,-11.504211,21.686119,-11.153907,-47.177877,-44.107757,-61.418976
VaR,0.165469,0.191149,-13.884539,20.202299,-13.526777,-47.117320,-45.710847,-62.447906
Mean-Std,0.055482,0.213707,-15.203729,22.927166,-14.915264,-52.918917,-50.650660,-68.867120
CVaR,0.159692,0.208512,-16.668162,22.248895,-16.363341,-53.267594,-51.742125,-67.476635


In [120]:
pd.options.display.float_format = '{:.2f}'.format
df_formatted = merged_df.set_index("model_name")
models_to_analyze = ["35000.pth_meanstd","15400.pth_cvar","11600.pth_weightedmean","27800.pth_var","18400.pth_meanvar",'Delta_agent', 'Gamma_agent','Vega_agent']
df_main = df_formatted.loc[models_to_analyze]
index_map = {
    "11600.pth_weightedmean": "Weighted Mean",
    "15400.pth_cvar": "CVaR",
    "18400.pth_meanvar": "Mean-CVaR",
    "35000.pth_meanstd": "Mean-Std",
    "27800.pth_var": "VaR",
    "Delta_agent": "Delta Hedging",
    "Gamma_agent": "Gamma Hedging",
    "Vega_agent": "Vega Hedging",
}
column_map = {
    "gamma_hr" : "Gamma Hedge Ratio",
    "vega_hr" : "Vega Hedge Ratio",
    "pnl_mean" : "Mean return",
    "pnl_std" : "Std", 
    "hed_cost" : "Transaction Cost", 
    "VaR_5" : "VaR95", 
    "CVaR_5" : "CVaR95", 
}
df_main = df_main.rename(index=index_map , columns = column_map )
df_main.index.name = "Objective Function"
df_main

,Gamma Hedge Ratio,Vega Hedge Ratio,Mean return,Std,Transaction Cost,meanstd,VaR95,CVaR95
Objective Function,,,,,,,,
Mean-Std,0.18,0.08,-5.24,11.99,-5.36,-24.95,-24.79,-33.35
CVaR,0.23,0.06,-5.00,11.95,-5.02,-24.65,-24.89,-33.17
Weighted Mean,0.19,0.07,-4.71,12.78,-4.87,-25.74,-25.38,-35.21
VaR,0.24,0.09,-6.13,12.03,-6.17,-25.92,-26.16,-35.00
Mean-CVaR,0.23,0.10,-6.16,12.44,-6.24,-26.63,-26.41,-34.83
Delta Hedging,-0.00,0.00,0.32,21.78,-0.00,-35.51,-34.42,-51.01
Gamma Hedging,1.00,0.16,-13.79,10.15,-13.69,-30.49,-30.76,-39.31
Vega Hedging,0.16,1.00,-26.16,23.57,-25.35,-64.94,-63.84,-85.89


In [121]:
latex_table = df_main.to_latex(
    float_format="%.2f",
    index=True,
    bold_rows=True,
    escape=False 
)
print(latex_table)

\begin{tabular}{lrrrrrrrr}
\toprule
 & Gamma Hedge Ratio & Vega Hedge Ratio & Mean return & Std & Transaction Cost & meanstd & VaR95 & CVaR95 \\
Objective Function &  &  &  &  &  &  &  &  \\
\midrule
\textbf{Mean-Std} & 0.18 & 0.08 & -5.24 & 11.99 & -5.36 & -24.95 & -24.79 & -33.35 \\
\textbf{CVaR} & 0.23 & 0.06 & -5.00 & 11.95 & -5.02 & -24.65 & -24.89 & -33.17 \\
\textbf{Weighted Mean} & 0.19 & 0.07 & -4.71 & 12.78 & -4.87 & -25.74 & -25.38 & -35.21 \\
\textbf{VaR} & 0.24 & 0.09 & -6.13 & 12.03 & -6.17 & -25.92 & -26.16 & -35.00 \\
\textbf{Mean-CVaR} & 0.23 & 0.10 & -6.16 & 12.44 & -6.24 & -26.63 & -26.41 & -34.83 \\
\textbf{Delta Hedging} & -0.00 & 0.00 & 0.32 & 21.78 & -0.00 & -35.51 & -34.42 & -51.01 \\
\textbf{Gamma Hedging} & 1.00 & 0.16 & -13.79 & 10.15 & -13.69 & -30.49 & -30.76 & -39.31 \\
\textbf{Vega Hedging} & 0.16 & 1.00 & -26.16 & 23.57 & -25.35 & -64.94 & -63.84 & -85.89 \\
\bottomrule
\end{tabular}

